# 10 — Search UI smoke test

Boots the FastAPI app in-process (TestClient) and hits the search endpoint with three canonical queries to verify the SPA contract.

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'apps').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')
from fastapi.testclient import TestClient
from apps.backend.api.main import app

client = TestClient(app)
QUERIES = ['唐律疏議中的十惡', '均田制度的內容', '安史之亂的影響']
results = []
for q in QUERIES:
    r = client.post('/api/search', json={'query': q, 'top_k': 5})
    results.append({'query': q, 'status': r.status_code, 'n_hits': len(r.json().get('results', []))})
print(json.dumps(results, ensure_ascii=False, indent=2))

ART = REPO_ROOT / 'notebooks' / '_artifacts' / '10_search_ui_smoke'
ART.mkdir(parents=True, exist_ok=True)
(ART / 'smoke.json').write_text(json.dumps({'ts': datetime.now(timezone.utc).isoformat(), 'results': results}, ensure_ascii=False, indent=2))
print('artifact:', ART / 'smoke.json')
